# 甄数：考卷 A「换掉任务描述」，在 Colab 上跑 Robometer

考卷 A 的 5 个数据集（dev 103 条、test 201 条），每条示范的俯视视频跑两次 Robometer：一次配原来的任务描述，一次配别的数据集的任务描述。一共 608 条曲线，结果是一堆小 JSON（缓存）。

**怎么跑**
1. 运行时类型选 **T4 GPU**。
2. 把本机新打的 `dist/assay-0.1.0-py3-none-any.whl` 拖进 `/content/`（同名，但内容是新的，要重新拖）。
3. 用左边「文件」面板的 Drive 按钮先挂好 Google Drive（上次已经授权过的话一点就好）。
4. 运行第 1 到 6 格。第 2 格下载约 1.6 GB 视频；5a 下载权重，下过的会秒过；5b 加载约 5 分钟，期间显存是 0、内存在涨，正常。第 6 格冒烟 2 对，会打印每条耗时和全量预计时间。
5. 没问题就运行第 7 格（全量，预计 40 到 60 分钟）。断了就从第 1 格重跑，算完的会跳过。
6. 跑完运行第 8 格，会下载 `exam_a_swap_cache.zip`。放到项目的 `C:\Claude_code\assay\cache\` 下解压，告诉 Claude。


In [ ]:
# 1. GPU、装依赖（lerobot 锁 0.6.1：适配器用到了它的内部方法）
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "lerobot[robometer]==0.6.1" psutil 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps /content/assay-0.1.0-py3-none-any.whl 2>&1 | tail -2
import importlib.metadata as md
print("lerobot", md.version("lerobot"), "assay", md.version("assay"))

In [ ]:
# 2. 考卷 A 配置（照抄本机 configs/exam_a.toml），下载关节数据和配置里那一路相机的视频（约 1.6 GB，有进度条）
import os
os.environ.pop("HF_HUB_DISABLE_XET", None)
CONFIG = "/content/exam_a.toml"
open(CONFIG, "w", encoding="utf-8").write("""# 考卷 A 用的公开数据集。一个数据集只属于一个切分；授权照抄数据卡。
# camera 是给视频评判用的那一路（俯视或侧视，看得到整个工作台）。
# 开发集定阈值；测试集换了任务（分拣、乐高块）、换了机型（SO-101、双臂），只跑一次。
[[dataset]]
repo = "lerobot/svla_so100_pickplace"   # 50 条，把方块放进盒子
split = "dev"
license = "apache-2.0"
camera = "observation.images.top"

[[dataset]]
repo = "lerobot/svla_so100_stacking"    # 56 条，红块叠到蓝块上
split = "dev"
license = "apache-2.0"
camera = "observation.images.top"

[[dataset]]
repo = "lerobot/svla_so100_sorting"     # 52 条，红蓝块分进左右两个盒子
split = "test"
license = "apache-2.0"
camera = "observation.images.top"

[[dataset]]
repo = "lerobot/svla_so101_pickplace"   # 50 条，乐高块放进透明盒；robot_type 写的是 so100_follower
split = "test"
license = "apache-2.0"
camera = "observation.images.up"

[[dataset]]
repo = "CoRL2026-CSI/SO101-DualArm-teleop_fold_towel_100epi"  # 100 条，双臂叠毛巾
split = "test"
license = "apache-2.0"
camera = "observation.images.left_top"
""")
from assay.eval.exam_a import load_config
from assay.eval.fetch import fetch
SPECS = load_config(CONFIG)
DATA = "/content/data"
for p in fetch(SPECS, DATA, with_videos=True):
    n = sum(1 for _ in p.rglob("*.mp4"))
    print(p.name, n, "个视频文件")

In [ ]:
# 3. 缓存放 Google Drive（和 RoboVAD 那次同一个目录，文件名不冲突）
import os
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
CACHE = "/content/drive/MyDrive/assay_cache"
os.makedirs(CACHE, exist_ok=True)
print("缓存目录", CACHE)

In [ ]:
# 4. 没挂 Drive 时的缓存位置
try:
    CACHE
except NameError:
    CACHE = "/content/cache"
    os.makedirs(CACHE, exist_ok=True)
print("缓存目录", CACHE)

In [ ]:
# 5a. 下载 Robometer 权重（8.9 GB，带进度条）；下好了再运行这格会秒过
import psutil
from huggingface_hub import snapshot_download
def ram():
    m = psutil.virtual_memory()
    return f"内存 已用 {(m.total - m.available) / 1e9:.1f} / {m.total / 1e9:.1f} GB"
print(snapshot_download("lerobot/Robometer-4B"))
print(ram())

In [ ]:
# 5b. 加载 Robometer（约 5 分钟）。这段时间显存一直是 0、内存在涨，是正常的
import gc, time
gc.collect()
print("加载前", ram(), flush=True)
from assay.judges.robometer import RobometerJudge
J = RobometerJudge()
t = time.time()
J._load()
gc.collect()
print("加载完", f"{time.time() - t:.0f}s", J.version, ram())

In [ ]:
# 6. 冒烟：dev 里前 2 对，看原描述和换后描述的进度差多少，估算全量时间
import time, torch
from assay.eval.task_swap import swap_pairs
from assay.judges.cache import judge_episode
PAIRS = {s: swap_pairs(SPECS, DATA, s) for s in ["dev", "test"]}
print({s: len(p) for s, p in PAIRS.items()}, "对")
per = []
for orig, swapped, cam in PAIRS["dev"][:2]:
    for ep in (orig, swapped):
        torch.cuda.reset_peak_memory_stats()
        t = time.time()
        c = judge_episode(J, ep, cam, CACHE, sample_fps=2.0, max_side=336)
        per.append(time.time() - t)
        print(ep.id, f"{per[-1]:.1f}s", f"峰值 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
        print("  任务", ep.task)
        print("  进度", [round(p, 2) for p in c.progress])
n = 2 * sum(len(p) for p in PAIRS.values())
print(f"全量 {n} 条预计 {sum(per) / len(per) * n / 3600:.1f} 小时")

In [ ]:
# 7. 全量：先 dev 再 test；断了重跑这一格，已算完的直接跳过
import gc
from tqdm.auto import tqdm
from assay.checks import NotApplicable
for split, pairs in PAIRS.items():
    jobs = [(ep, cam) for o, s, cam in pairs for ep in (o, s)]
    skipped = 0
    for ep, cam in tqdm(jobs, desc=split):
        try:
            judge_episode(J, ep, cam, CACHE, sample_fps=2.0, max_side=336)
        except NotApplicable as e:
            skipped += 1; print("跳过", ep.id, e)
        except torch.OutOfMemoryError:
            skipped += 1; print("显存不够，跳过", ep.id)
            gc.collect(); torch.cuda.empty_cache()
    print(split, "完成", len(jobs) - skipped, "跳过", skipped, flush=True)

In [ ]:
# 8. 只打包考卷 A 的缓存，压缩包里带 robometer/ 这一层目录
import zipfile
from pathlib import Path
from google.colab import files
prefixes = tuple(Path(s.repo.replace("/", "__")).name for s in SPECS)
src = Path(CACHE) / "robometer"
picked = [f for f in src.iterdir() if f.name.startswith(prefixes)]
with zipfile.ZipFile("/content/exam_a_swap_cache.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in picked:
        z.write(f, f"robometer/{f.name}")
print(len(picked), "个文件")
files.download("/content/exam_a_swap_cache.zip")